# Koschei Sentinel — Qwen3.5 0.8B Cyber SFT Micro-Smoke (Kaggle)

This is the **first real gradient-update proof run**. It uses the same Defense Reflex v3 supervision, text-only Qwen3.5 executor, receipt, attestation, and portable export verifier as the 9B smoke run.

Before running, set Kaggle Notebook **Accelerator = GPU** and **Internet = ON**. The produced adapter is deliberately `promotion_eligible=false`.

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/koschei-sentinel')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/bugsbuny243/koschei-sentinel.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
print('Commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import os, subprocess
env = os.environ.copy()
env['KOSCHEI_KAGGLE_OUTPUT_ROOT'] = '/kaggle/working/koschei-sentinel-micro-output'
env['HF_HOME'] = '/kaggle/working/hf-cache'
subprocess.run(['bash', str(repo / 'scripts/run_cyber_sft_qwen35_08b_micro_kaggle.sh'), str(repo)], check=True, env=env)


In [ ]:
import json
from pathlib import Path

out = Path('/kaggle/working/koschei-sentinel-micro-output')
receipt = json.loads((out / 'run' / 'training-receipt.json').read_text())
attestation = json.loads((out / 'run-attestation.json').read_text())
export_verification = json.loads((out / 'export-verification.json').read_text())
preflight = json.loads((out / 'model-preflight.json').read_text())
warning_path = out / 'run' / 'runtime-warnings.json'
warnings = json.loads(warning_path.read_text())['warnings'] if warning_path.exists() else []

assert preflight['resolved_revision'] == 'dc7cdfe2ee4154fa7e30f5b51ca41bfa40174e68'
assert receipt['global_step'] > 0
assert attestation['selected_profile'] == 'micro'
assert export_verification['valid'] is True

print('MODEL:', receipt['base_model'])
print('REVISION:', receipt['base_revision'])
print('GLOBAL STEP:', receipt['global_step'])
print('GPU:', receipt['cuda_device_name'])
print('TOTAL VRAM GiB:', round(receipt['cuda_total_memory_gb'], 3))
print('PEAK ALLOCATED GiB:', round(receipt['max_cuda_memory_allocated_gb'], 3))
print('PEAK RESERVED GiB:', round(receipt['max_cuda_memory_reserved_gb'], 3))
print('ADAPTER SHA256:', receipt['adapter_digest'])
print('ATTESTATION SHA256:', attestation['attestation_sha256'])
print('PORTABLE EXPORT VALID:', export_verification['valid'])
print('RUNTIME WARNINGS:', warnings or 'none')
print('ZIP: /kaggle/working/koschei-sentinel-qwen35-08b-micro.zip')
